---
title: "Data Munging + AI Gotchas"
execute:
  enabled: true
jupyter: python3
---

## When clean-looking code gives wrong answers

You hand the hospital readmissions dataset to an AI assistant and ask: "Which hospitals have the worst readmission rates?" It returns clean Python code, professional-looking plots, and a confident ranking. The code runs without errors. The answer is wrong.

In Chapter 1, we introduced the hospital readmissions dataset and its consequential decisions. In Chapter 2, we explored the data and found the mess. Now we roll up our sleeves.

In this chapter we get our hands dirty with the unglamorous work of **data munging**: **joins**, missing data handling, type conversions, and all the decisions that happen *before* analysis. By the end, we'll show you an AI analysis that looks professional, produces clean code and nice plots — and gets the answer dangerously wrong.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
plt.rcParams['font.size'] = 12

# Load data
DATA_DIR = 'data'

## Part 1: Joining datasets

We have two hospital files:
- `hrrp_full.csv` — readmission data (18,000+ rows, one per hospital-condition pair)
- `hospital_info.csv` — hospital characteristics (ownership, type, star rating, etc.)

To ask richer questions — like "do nonprofit hospitals have better readmission rates?" — we need to **join** them.

In [ ]:
# Load both datasets
hrrp = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hrrp_full.csv')
hospital_info = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hospital_info.csv')

# Ensure Facility ID types match (hospital_info may have zero-padded strings)
hospital_info['Facility ID'] = pd.to_numeric(hospital_info['Facility ID'], errors='coerce')

print(f"HRRP:          {hrrp.shape[0]:,} rows, {hrrp.shape[1]} columns")
print(f"Hospital Info:  {hospital_info.shape[0]:,} rows, {hospital_info.shape[1]} columns")
print()
print(f"HRRP unique hospitals:      {hrrp['Facility ID'].nunique():,}")
print(f"Hospital Info unique IDs:   {hospital_info['Facility ID'].nunique():,}")

The **join key** is `Facility ID` — the column used to match rows across tables. But the two datasets have different numbers of unique hospitals, so not every ID appears in both. Two common join strategies handle this mismatch differently:

- An **inner join** keeps only rows whose key appears in *both* tables, discarding the rest.
- A **left join** keeps every row from the left table and attaches matching data from the right table. Where no match exists, the right-side columns become NaN.

:::{.callout-tip}
## Think About It
Before we run the join — predict: will the inner join have more rows, fewer rows, or the same number as the HRRP table? What about the left join?
:::

In [ ]:
# Inner join: only hospitals that appear in BOTH datasets
inner = pd.merge(hrrp, hospital_info[['Facility ID', 'Hospital Type', 'Hospital Ownership',
                                       'Hospital overall rating', 'Emergency Services']],
                 on='Facility ID', how='inner')
print(f"Inner join: {inner.shape[0]:,} rows")
print(f"  Unique hospitals: {inner['Facility ID'].nunique():,}")

# Left join: keep ALL hrrp rows, add hospital info where available
left = pd.merge(hrrp, hospital_info[['Facility ID', 'Hospital Type', 'Hospital Ownership',
                                      'Hospital overall rating', 'Emergency Services']],
                on='Facility ID', how='left')
print()
print(f"Left join:  {left.shape[0]:,} rows")
print(f"  Unique hospitals: {left['Facility ID'].nunique():,}")
print(f"  Rows with missing Hospital Type: {left['Hospital Type'].isna().sum():,}")

### Inner vs left join: what gets dropped?

The inner join dropped some hospitals that appear in the readmissions data but not in the hospital info table (or vice versa). The left join kept all readmission records but left hospital characteristics as NaN where no match was found.

**The join type is a decision, not a default.** Every choice gives a different answer. An inner join used without checking may silently lose data.

:::{.callout-important}
## Key Principle
After every join, check your row counts. Did you gain rows (duplicates in the join key)? Lose rows (unmatched keys)? Either one is a signal worth investigating.
:::

In [ ]:
# What hospitals are in HRRP but NOT in hospital_info?
hrrp_ids = set(hrrp['Facility ID'].unique())
info_ids = set(hospital_info['Facility ID'].unique())

only_in_hrrp = hrrp_ids - info_ids
only_in_info = info_ids - hrrp_ids

print(f"Hospitals only in HRRP (no info available):     {len(only_in_hrrp)}")
print(f"Hospitals only in Hospital Info (no readmissions): {len(only_in_info)}")

The bar chart below shows the row counts side by side — notice how the inner join shrinks the dataset.

In [ ]:
# Visualize: how many rows survive each join type?
join_types = ['HRRP\n(original)', 'Inner join', 'Left join']
join_counts = [len(hrrp), inner.shape[0], left.shape[0]]

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(join_types, join_counts, color=['steelblue', '#e74c3c', '#2ecc71'], edgecolor='white')
ax.axhline(y=len(hrrp), color='steelblue', linestyle='--', alpha=0.5, label='Original HRRP rows')
ax.set_ylabel('Number of rows')
ax.set_title('Row counts by join type')
for bar, count in zip(bars, join_counts):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'{count:,}', ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

We'll use the left join going forward, so we don't lose any readmission data. The hospitals without matching info will simply have NaN for characteristics like ownership and star rating. (Missing values can sometimes be turned into useful *features* — for example, an indicator column for "was this value missing?" can carry signal. Tree-based models (Chapter 7) can also handle missing data natively, without requiring imputation.)

## Part 2: Missing data handling

In Chapter 2, we saw that about 15% of the readmission data has "Too Few to Report" instead of actual numbers. Now let's see what happens when we handle this in different ways.

In [ ]:
# Work with heart failure data (most complete condition)
merged = left.copy()
hf = merged[merged['Measure Name'] == 'READM-30-HF-HRRP'].copy()

print(f"Heart failure rows: {len(hf):,}")
print(f"Missing Excess Readmission Ratio: {hf['Excess Readmission Ratio'].isna().sum():,}")
print(f"Missing Number of Discharges: {hf['Number of Discharges'].isna().sum():,}")

### Three approaches to missing data

Let's compute the average excess readmission ratio three different ways and see how the answers differ.

In [ ]:
# Approach 1: Drop rows with missing values
approach1 = hf.dropna(subset=['Excess Readmission Ratio'])
mean1 = approach1['Excess Readmission Ratio'].mean()
n1 = len(approach1)

# Approach 2: Fill missing values with the column mean
mean_val = hf['Excess Readmission Ratio'].mean()  # mean of non-missing
approach2 = hf.copy()
approach2['Excess Readmission Ratio'] = approach2['Excess Readmission Ratio'].fillna(mean_val)
mean2 = approach2['Excess Readmission Ratio'].mean()
n2 = len(approach2)

# Approach 3: Leave as NaN (compute mean of available data only)
mean3 = hf['Excess Readmission Ratio'].mean()  # pandas ignores NaN by default
n3 = hf['Excess Readmission Ratio'].notna().sum()

print("Average Excess Readmission Ratio for Heart Failure:")
print(f"  Approach 1 (drop missing):    {mean1:.4f}  (n = {n1:,})")
print(f"  Approach 2 (fill with mean):  {mean2:.4f}  (n = {n2:,})")
print(f"  Approach 3 (ignore NaN):      {mean3:.4f}  (n = {n3:,})")

Approaches 1 and 3 produce the same result — pandas ignores NaN by default, so you may already be dropping data without realizing it. Approach 2 (**mean imputation**) also gives the same mean, by construction: replacing missing values with the mean does not shift the average. However, mean imputation does shrink the sample variance, since every imputed value sits exactly at the center of the distribution rather than reflecting the true spread. Mean imputation also distorts correlations between variables, pulling genuine relationships toward zero.

You might be thinking: if they all give the same answer, who cares? Here's where it matters — when we compare groups.

:::{.callout-tip}
## Think About It
If we drop all rows with missing readmission data, which types of hospitals do you think we lose the most of? Why?
:::

In [ ]:
# Are the missing values evenly distributed across ownership types?
# .count() excludes NaN, .size() includes it — the difference is the missing count
ownership_missing = hf.groupby('Hospital Ownership')['Excess Readmission Ratio'].agg(['size', 'count'])
ownership_missing.columns = ['total', 'non_missing']
ownership_missing['missing'] = ownership_missing['total'] - ownership_missing['non_missing']
ownership_missing['% missing'] = (ownership_missing['missing'] / ownership_missing['total'] * 100).round(1)
ownership_missing = ownership_missing.sort_values('% missing', ascending=False)
ownership_missing

The table reveals large differences. A horizontal bar chart makes the disparity more vivid.

In [ ]:
# Visualize: missingness rate by ownership type
fig, ax = plt.subplots(figsize=(10, 5))
ownership_missing['% missing'].plot.barh(ax=ax, color='coral', edgecolor='white')
ax.set_xlabel('% of rows with missing Excess Readmission Ratio')
ax.set_title('Missing data is NOT evenly distributed across hospital types')
ax.invert_yaxis()
plt.tight_layout()
plt.show()

The missing data is *not* evenly distributed across ownership types. Some types of hospitals are far more likely to have missing data. Dropping missing rows changes the mix of hospitals in the analysis — a phenomenon called **informative missingness**, where the absence of data itself carries a signal about the hospital.

A vivid example from another domain: in ambulance data, a missing heart rate often means the EMTs were too busy saving the patient's life to record vitals. The missing value *is* the signal — the opposite of random missingness.

:::{.callout-important}
## Definition: Missing Data Patterns
Statisticians distinguish three patterns:

- **Missing Completely At Random (MCAR)** — missingness is unrelated to any variable. Example: a lab technician accidentally drops a test tube, destroying the sample.
- **Missing At Random (MAR)** — missingness depends on observed variables but not on the missing value itself. Example: younger patients are less likely to have cholesterol recorded, but among patients of the same age, missingness does not depend on the cholesterol level.
- **Missing Not At Random (MNAR)** — missingness depends on the unobserved value itself or on other unobserved variables. Example: patients with very high blood pressure skip follow-up appointments *because* they feel unwell.

Hospital readmission data falls into the MNAR category — small hospitals cannot report precisely *because* their volumes are low, and volume relates to the outcome. MNAR is the hardest pattern to handle. Every strategy introduces bias; the question is which bias you can live with.
:::

## Part 3: Data types gone wrong

Missing data is one kind of data quality problem. But even present data can trick you when the *type* is wrong. Numbers stored as strings, dates that aren't parsed, categories encoded as integers — each creates a silent trap. Let's see some examples.

In [ ]:
# "Number of Readmissions" is stored as a string because of "Too Few to Report"
print("Type of 'Number of Readmissions':", hrrp['Number of Readmissions'].dtype)
print()

# If we try to compute the mean directly, it fails
try:
    hrrp['Number of Readmissions'].mean()
except TypeError as e:
    print(f"Error: {e}")

The fix is `pd.to_numeric` with `errors='coerce'`, which turns unparseable entries into NaN.

In [ ]:
# Convert to numeric, coercing non-numeric values to NaN
hrrp['Readmissions_numeric'] = pd.to_numeric(hrrp['Number of Readmissions'], errors='coerce')
print(f"Successfully converted. Mean readmissions: {hrrp['Readmissions_numeric'].mean():.1f}")
print(f"Rows lost to coercion: {hrrp['Readmissions_numeric'].isna().sum() - hrrp['Number of Readmissions'].isna().sum():,}")

Look at how many rows were lost to coercion. Those are rows where "Too Few to Report" was silently converted to NaN. If you're not paying attention, this happens invisibly.

Now let's look at the Airbnb data for another common type problem.

In [ ]:
# Airbnb price data
airbnb = pd.read_csv(f'{DATA_DIR}/airbnb/listings.csv', low_memory=False,
                     usecols=['price', 'weekly_price', 'monthly_price', 'cleaning_fee',
                              'bedrooms', 'bathrooms', 'room_type',
                              'neighbourhood_group_cleansed', 'number_of_reviews'])

print("Price column dtype:", airbnb['price'].dtype)
print("Sample values:", airbnb['price'].head().tolist())

Depending on how the CSV was prepared, the `price` column may already be numeric (int or float) or may contain dollar signs and commas that force pandas to read it as a string. Always check the dtype before proceeding. If the column is already numeric, no cleaning is needed. If it contains formatting characters, strip them first.

In [ ]:
# If price is stored as a string with "$" and "," characters, clean it:
if airbnb['price'].dtype == 'object':
    airbnb['price_clean'] = (airbnb['price'].astype(str)
                             .str.replace('[$,]', '', regex=True)
                             .astype(float))
    print(f"Converted from string. dtype: {airbnb['price_clean'].dtype}")
else:
    airbnb['price_clean'] = airbnb['price'].astype(float)
    print(f"Already numeric. dtype: {airbnb['price_clean'].dtype}")

print(f"Mean price: ${airbnb['price_clean'].mean():.2f}")

The broader lesson: always inspect dtypes before writing cleaning code. Applying string operations to a column that is already numeric will either fail or produce nonsense.

### When data needs cleaning

Real-world CSVs are rarely tidy. Here's what raw Airbnb price data looks like before cleaning:

In [ ]:
# Simulating raw price data — what Inside Airbnb's CSV sometimes looks like
raw_prices = pd.Series(['$1,200.00', '$85.00', '$250.00', '$0.00', 'unknown', 'NA', '-50', 'Inf', None])
print("Raw format:")
print(raw_prices)
print(f"\nType: {raw_prices.dtype}")
# raw_prices.mean()  # Would raise TypeError — can't average strings!

The fix: replace non-numeric placeholders with `NaN`, strip formatting characters, convert to numeric with `pd.to_numeric()`, and handle invalid values like negatives.

In [ ]:
# Step 1: Replace non-numeric placeholders with NaN
cleaned = raw_prices.replace(['unknown', 'NA', 'Inf', '-Inf'], np.nan)
# Step 2: Strip dollar signs and commas
cleaned = cleaned.str.replace(r'[\$,]', '', regex=True)
# Step 3: Convert to numeric, coercing remaining problems to NaN
cleaned = pd.to_numeric(cleaned, errors='coerce')
# Step 4: Replace negative values with NaN (prices can't be negative)
cleaned = cleaned.where(cleaned >= 0)
print("After cleaning:")
print(cleaned)
print(f"\nValid prices: {cleaned.notna().sum()} out of {len(cleaned)}")
print(f"Mean of valid: ${cleaned.mean():.2f}")

Always check `df.dtypes` early. A column that *looks* numeric may be stored as strings — and pandas won't warn you until you try to compute with it.

:::{.callout-tip}
## Think About It
The Airbnb dataset has 25 listings at $0/night. Are those real prices, data entry errors, or something else? How would you decide?
:::

### The "Hospital overall rating" trap

Here's a subtler type problem. The hospital star rating column looks numeric (1-5), but...

In [ ]:
# What values does the star rating take?
print("Hospital overall rating — value counts:")
print(hospital_info['Hospital overall rating'].value_counts().sort_index())

"Not Available" is mixed in with the numbers. If you convert this to numeric without checking, those hospitals get NaN. If you label-encode it (1, 2, 3, 4, 5, 6), you've just told your model that "Not Available" is better than 5 stars.

:::{.callout-warning}
## Type Errors Are Silent
These type issues seem trivial. They're not. They're the #1 source of silent errors in data analysis.
:::

## Part 4: AI Gotchas — When the machine is confidently wrong

> *"The combination of some data and an aching desire for an answer does not ensure that a reasonable one can be extracted from a given body of data."*
> — John Tukey

Let's simulate what happens when you hand an AI assistant the hospital data and say: *"Find the hospitals with the highest readmission rates."*

Here are four common mistakes — each one produces plausible-looking output while introducing serious error.

### Gotcha #1: Silent row dropping

A common first step is to call `.dropna()` to "clean" the data. Let's see the impact.

In [ ]:
# A common "cleaning" step: drop all rows with any NaN
hrrp_full = pd.read_csv(f'{DATA_DIR}/hospital-readmissions/hrrp_full.csv')

before = len(hrrp_full)
hrrp_clean = hrrp_full.dropna()
after = len(hrrp_clean)

print(f"Before dropna(): {before:,} rows")
print(f"After dropna():  {after:,} rows")
print(f"Dropped: {before - after:,} rows ({(before - after)/before*100:.1f}%)")

Where did those rows go? Comparing the state distribution before and after reveals that the drop is uneven. (Pandas provides `.plot.bar()` as a shorthand for bar charts on pre-computed Series or DataFrames — a convenient alternative to seaborn's `sns.barplot()` when the data is already aggregated.)

In [ ]:
# What changed? Compare the distribution of hospitals by state
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# Before: top 10 states by number of records
hrrp_full['State'].value_counts().head(10).plot.bar(ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title(f'Top 10 States — Before Dropping ({before:,} rows)')
axes[0].set_ylabel('Records')
axes[0].set_xlabel('State')

# After: top 10 states
hrrp_clean['State'].value_counts().head(10).plot.bar(ax=axes[1], color='coral', edgecolor='white')
axes[1].set_title(f'Top 10 States — After Dropping ({after:,} rows)')
axes[1].set_ylabel('Records')
axes[1].set_xlabel('State')

plt.tight_layout()
plt.show()

The state distribution shifts. States with many small or rural hospitals lose a larger fraction of their data. The result is **selection bias**: dropping rows with missing data changes *which* hospitals remain in the sample, and the AI proceeds with the "cleaned" data as if nothing happened.

### Gotcha #2: Wrong ranking metric

:::{.callout-tip}
## Think About It
If you were ranking hospitals by readmission performance, which column would you use — `Predicted Readmission Rate` or `Excess Readmission Ratio`? Why?
:::

An analyst unfamiliar with CMS methodology might rank hospitals by `Predicted Readmission Rate`, the most intuitive-sounding column. But the Predicted Rate reflects both patient severity and hospital performance. To isolate hospital quality, CMS computes the `Excess Readmission Ratio`, which adjusts for patient mix. Choosing the right metric requires domain knowledge that no amount of data cleaning can substitute for.

In [ ]:
# Compare: AI's naive ranking vs correct ranking (Heart Failure)
hf_data = hrrp_full[hrrp_full['Measure Name'] == 'READM-30-HF-HRRP'].dropna(
    subset=['Predicted Readmission Rate', 'Excess Readmission Ratio'])

# Naive answer: rank by predicted rate
naive_top20 = set(hf_data.nlargest(20, 'Predicted Readmission Rate')['Facility ID'])

# Correct answer: rank by excess ratio
adjusted_top20 = set(hf_data.nlargest(20, 'Excess Readmission Ratio')['Facility ID'])

overlap = naive_top20 & adjusted_top20
print(f"Top 20 by Predicted Rate vs Top 20 by Excess Ratio:")
print(f"  Overlap: {len(overlap)} hospitals appear in both lists")
print(f"  {20 - len(overlap)} hospitals are in one list but not the other")
print()
print("The naive ranking disagrees substantially with the risk-adjusted ranking.")

The scatter plot below maps each hospital by both metrics. Hospitals flagged by one ranking but not the other appear in distinct colors.

In [ ]:
# Visualize: how different are the two rankings?
fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(hf_data['Predicted Readmission Rate'], hf_data['Excess Readmission Ratio'],
           alpha=0.3, s=15, color='gray', label='All hospitals')

# Highlight the top 20 by each metric
naive_mask = hf_data['Facility ID'].isin(naive_top20)
adjusted_mask = hf_data['Facility ID'].isin(adjusted_top20)
both_mask = hf_data['Facility ID'].isin(overlap)

ax.scatter(hf_data.loc[naive_mask & ~both_mask, 'Predicted Readmission Rate'],
           hf_data.loc[naive_mask & ~both_mask, 'Excess Readmission Ratio'],
           color='coral', s=40, label='Top 20 by Predicted Rate only', zorder=5)
ax.scatter(hf_data.loc[adjusted_mask & ~both_mask, 'Predicted Readmission Rate'],
           hf_data.loc[adjusted_mask & ~both_mask, 'Excess Readmission Ratio'],
           color='steelblue', s=40, label='Top 20 by Excess Ratio only', zorder=5)
ax.scatter(hf_data.loc[both_mask, 'Predicted Readmission Rate'],
           hf_data.loc[both_mask, 'Excess Readmission Ratio'],
           color='gold', s=60, edgecolor='black', label='In both top-20 lists', zorder=6)

ax.set_xlabel('Predicted Readmission Rate')
ax.set_ylabel('Excess Readmission Ratio')
ax.set_title('Which hospitals are "worst"? It depends on the metric.')
ax.legend(fontsize=9, loc='upper left')
plt.tight_layout()
plt.show()

### Gotcha #3: Data leakage

:::{.callout-tip}
## Think About It
Look at the columns in the hospital data: `Predicted Readmission Rate`, `Expected Readmission Rate`, `Excess Readmission Ratio`. If you wanted to predict Excess Readmission Ratio, which columns would be dangerous to include as inputs?
:::

Here's a subtler trap. Suppose the AI builds a model to predict the Excess Readmission Ratio. It would naturally include all numeric columns as features — including `Expected Readmission Rate`.

:::{.callout-important}
## Definition: Data Leakage
**Data leakage** occurs when your model's input features contain information that wouldn't be available at prediction time, or that is mathematically derived from the target. Here, **Excess Ratio = Predicted / Expected**, so including Expected Rate as a feature is leakage — the feature contains the answer by construction, like using tomorrow's stock price to "predict" today's return.
:::

The leakage is visible directly in the correlations. Recall that **correlation** measures the strength of the linear relationship between two variables, ranging from -1 (perfect negative) to +1 (perfect positive), with 0 indicating no linear relationship. We will formalize correlation in Chapter 4.

In [ ]:
# The leakage is visible as near-perfect correlation
hf_model = hf_data[['Number of Discharges', 'Predicted Readmission Rate',
                     'Expected Readmission Rate', 'Excess Readmission Ratio']].dropna()

print("Correlation with Excess Readmission Ratio:")
print(f"  Expected Readmission Rate:  {hf_model['Expected Readmission Rate'].corr(hf_model['Excess Readmission Ratio']):.4f}")
print(f"  Predicted Readmission Rate: {hf_model['Predicted Readmission Rate'].corr(hf_model['Excess Readmission Ratio']):.4f}")
print(f"  Number of Discharges:       {hf_model['Number of Discharges'].corr(hf_model['Excess Readmission Ratio']):.4f}")

Expected Rate is almost perfectly correlated with Excess Ratio — unsurprising, given that one is computed from the other. Any model that includes Expected Rate as a feature would appear to perform brilliantly while simply encoding a tautology. Number of Discharges, by contrast, shows the kind of weak correlation you would expect from a genuinely independent variable. We will study regression properly in Chapter 5, but the leakage problem is conceptual: the feature contains the answer by construction.

### Gotcha #4: Treating categoricals as numbers

It is tempting to treat `Hospital overall rating` as numeric (1, 2, 3, 4, 5). But some values are "Not Available." And even the numeric values represent an ordinal scale, not a continuous one — the difference between 4-star and 5-star is not necessarily the same as between 1-star and 2-star.

In [ ]:
# Merge star ratings into readmission data
hf_with_info = pd.merge(
    hf_data, hospital_info[['Facility ID', 'Hospital overall rating']],
    on='Facility ID', how='left')

# Convert to numeric, silently dropping "Not Available"
# (Same errors='coerce' pattern we've seen twice — non-numeric values become NaN)
hf_with_info['rating_numeric'] = pd.to_numeric(
    hf_with_info['Hospital overall rating'], errors='coerce')

print(f"Hospitals with numeric rating: {hf_with_info['rating_numeric'].notna().sum():,}")
print(f"Hospitals with 'Not Available': {(hf_with_info['Hospital overall rating'] == 'Not Available').sum():,}")
print(f"Hospitals dropped by coercion: {hf_with_info['rating_numeric'].isna().sum():,}")
print()

# Show the relationship the AI would find
rating_means = hf_with_info.groupby('rating_numeric')['Excess Readmission Ratio'].agg(['mean', 'count'])
rating_means.columns = ['Mean Excess Ratio', 'Count']
print("Average Excess Readmission Ratio by Star Rating:")
print(rating_means.to_string())

There *is* a relationship between star rating and readmission performance — lower-rated hospitals tend to have higher excess ratios. But notice that "Not Available" hospitals (often small, rural, or specialized) were silently excluded from this analysis. Those excluded hospitals might tell a different story.

## Key Takeaways

- **Every cleaning choice is a decision.** Dropping rows, imputing values, and converting types all change which records enter your analysis — and can change your conclusions.
- **Joins and missing data reshape your sample.** Always check row counts before and after. Informative missingness means the pattern of what is absent carries its own signal.
- **Domain knowledge has no substitute.** Choosing the right metric, spotting leakage, and interpreting missingness all require understanding the data-generating process — not just the data itself.
- **Verify, then trust.** After every cleaning step, ask: did I gain or lose rows? Did any group lose more than others? Do column types match their intended use?

## Study guide

### Key ideas

- **Data munging / wrangling** — the process of cleaning, transforming, and preparing raw data for analysis. Every cleaning choice is a decision that changes your conclusions — there is no "neutral" default.
- **Inner join** — combines two tables, keeping only rows with matching keys in *both* tables. **Left join** — keeps all rows from the left table and adds matched data from the right (NaN where no match). The **join key** is the column(s) used to match rows.
- After every join, check row counts: gaining rows means duplicate keys; losing rows means unmatched keys.
- **Mean imputation** — filling missing values with the column mean; preserves the mean but shrinks variance and distorts relationships.
- **Selection bias** — systematic difference between the sample you analyze and the population you care about, caused here by dropping rows.
- **Informative missingness** — when the fact that data is missing tells you something about the value (e.g., small hospitals can't report). Three patterns: **MCAR** (unrelated to any variable), **MAR** (depends on observed variables only), **MNAR** (depends on the unobserved value itself) — MNAR is the hardest to handle.
- Missing data patterns are often informative: *which* data is missing can be as important as the values themselves.
- **Data leakage** — including information in your model that wouldn't be available at prediction time, or that is mathematically derived from the target.
- AI assistants produce plausible-looking analyses but systematically fail to question their own data-cleaning decisions.

### Computational tools

- `pd.merge(left, right, on=..., how=...)` — join two DataFrames; `how` controls inner/left/right/outer
- `pd.to_numeric(series, errors='coerce')` — convert strings to numbers, turning failures into NaN
- `.dropna(subset=[...])` — drop rows with NaN in specified columns
- `.fillna(value)` — fill NaN with a specified value
- `.isna()` / `.notna()` — boolean mask for missing / non-missing values
- `.value_counts()` — count unique values (useful for spotting unexpected entries like "Not Available")
- `.str.replace(pat, repl, regex=True)` — apply string operations element-wise on a Series (also `.str.lower()`, `.str.strip()`, `.str.contains()`, etc.)
- `.groupby(col).agg(...)` — split data by groups and compute summary statistics; `.agg()` accepts functions like `'mean'`, `'count'`, `'sum'`, or lists/dicts of them
- `.astype(dtype)` — cast a Series to a specified type (e.g., `float`, `int`, `str`, `'category'`)

### For the quiz

You should be able to: (1) predict the row count after an inner vs. left join given the key overlap, (2) explain why dropping missing data can introduce selection bias, (3) identify data leakage when a feature is mathematically related to the target, (4) recognize when a column's dtype doesn't match its intended use.